# Frequency Calibration: Update ZPL Distribution CSV

The wavemeter was not running during the ZPL distribution measurement, so all frequency values
in the CSV are wrong (all the same). This notebook:

1. Loads the existing CSV
2. Steps the laser to each measured voltage via `dl_pro.set_pc_voltage()`
3. Reads the wavemeter at each point
4. Recomputes the GHz frequency relative to a reference
5. Writes the corrected CSV back to disk

## 1 · Configuration

In [8]:
import time
import numpy as np
import pandas as pd

# ── Path to the CSV that needs to be updated ──────────────────────────────
CSV_PATH = r"z:\Vlad\Daily_data\attodry\2026\03\2026-03-25\zpl_distribution_logic\JK-waeguides-second-anneal20k-udpated-manual - Copy\distribution_histogram.csv"  # <-- set this!

# ── Reference frequency (same as ZPLDistributionLogic._zero_frequency) ───
ZERO_FREQ_THz = 484.135   # THz  (change to match your measurement)

# ── Settling time after moving the laser voltage ──────────────────────────
SETTLE_TIME_s = 2.0        # seconds

# ── Wavemeter retry settings ──────────────────────────────────────────────
WM_MAX_RETRIES = 10
WM_RETRY_DELAY = 0.1       # seconds between retries

# ── qudi module handles ────────────────────────────────────────────────────
laser  = dl_pro          # SimpleLaserInterface: has set_pc_voltage()
wm     = ws_wavemeter    # wavemeter: has get_current_wavelength() -> nm

print("Config OK")

Config OK


## 2 · Load existing CSV

In [9]:
# encoding='latin-1' handles the µ character (0xb5) written by Windows apps
df = pd.read_csv(CSV_PATH, encoding='latin-1')
print(f"Loaded {len(df)} rows from {CSV_PATH}")
print(df.head(10).to_string())

# The unique voltages we need to sweep
voltages = sorted(df["Voltage (V)"].unique())
print(f"\nUnique voltages ({len(voltages)}): {voltages}")

ParserError: Error tokenizing data. C error: Expected 1 fields in line 6, saw 5


## 3 · Helper: read wavemeter with retry

In [ ]:
def read_frequency_ghz(wm_module, zero_thz=ZERO_FREQ_THz,
                       max_retries=WM_MAX_RETRIES, retry_delay=WM_RETRY_DELAY):
    """
    Read the current wavemeter wavelength and return frequency in GHz
    relative to zero_thz.

    Returns np.nan if no valid reading is obtained.
    """
    val = 0.0
    for attempt in range(max_retries):
        try:
            if hasattr(wm_module, 'get_current_wavelength'):
                val = float(wm_module.get_current_wavelength())
            elif hasattr(wm_module, 'get_wavelength'):
                val = float(wm_module.get_wavelength())
        except Exception:
            val = 0.0

        # If zero, try explicit channels 1-8
        if val == 0 and hasattr(wm_module, 'get_wavelength'):
            for ch in range(1, 9):
                try:
                    tmp = float(wm_module.get_wavelength(ch))
                    if tmp > 0:
                        val = tmp
                        break
                except Exception:
                    pass

        if val > 0:
            break
        time.sleep(retry_delay)

    if val <= 0:
        return np.nan

    # Convert: if val > 550 it's in nm, otherwise treat as THz
    freq_thz = 299792.458 / val if val > 550 else val
    return (freq_thz - zero_thz) * 1000.0   # GHz relative to reference


# Quick test
print("Current wavemeter reading:", read_frequency_ghz(wm), "GHz")

## 4 · Sweep voltages and record frequencies

In [ ]:
voltage_to_freq = {}   # maps voltage -> measured GHz

print(f"Sweeping {len(voltages)} voltage points...\n")

for i, v in enumerate(voltages):
    # Move laser
    laser.set_pc_voltage(v)
    time.sleep(SETTLE_TIME_s)

    # Read wavemeter
    freq_ghz = read_frequency_ghz(wm)
    voltage_to_freq[v] = freq_ghz

    status = f"{freq_ghz:.4f} GHz" if not np.isnan(freq_ghz) else "NO READING"
    print(f"  [{i+1:3d}/{len(voltages)}] V={v:6.1f} V  ->  {status}")

print("\nSweep complete.")

## 5 · Inspect the calibration curve

In [ ]:
import matplotlib.pyplot as plt

volts = list(voltage_to_freq.keys())
freqs = [voltage_to_freq[v] for v in volts]

plt.figure(figsize=(9, 4))
plt.plot(volts, freqs, 'o-', ms=4)
plt.xlabel("Voltage (V)")
plt.ylabel("Frequency (GHz)")
plt.title("Voltage -> Frequency calibration")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary table
print(f"{'Voltage (V)':>12}  {'Old GHz':>14}  {'New GHz':>14}")
print("-" * 44)
old_freq = df.groupby("Voltage (V)")["Frequency (GHz)"].first()
for v in volts:
    old = old_freq.get(v, np.nan)
    new = voltage_to_freq[v]
    print(f"{v:>12.1f}  {old:>14.4f}  {new:>14.4f}")

## 6 · Update the DataFrame and save

In [ ]:
# Map each row's voltage to its newly measured frequency
df["Frequency (GHz)"] = df["Voltage (V)"].map(voltage_to_freq)

# Rows with no wavemeter reading (NaN) are flagged
n_nan = df["Frequency (GHz)"].isna().sum()
if n_nan:
    print(f"Warning: {n_nan} rows have NaN frequency (wavemeter returned no signal at those voltages).")

# Save with utf-8 encoding (safe for all characters)
OUT_PATH = CSV_PATH.replace(".csv", "_calibrated.csv")
df.to_csv(OUT_PATH, index=False, encoding='utf-8')
print(f"\nSaved calibrated data to:\n  {OUT_PATH}")
df.head(10)

## 7 · (Optional) Reload into zpl_distribution_logic

If the `zpl_distribution_logic` module is running in qudi you can push the corrected
frequencies directly into its in-memory state so the GUI plot updates without restarting.

In [ ]:
# Update the live logic state (optional)
logic = zpl_distribution_logic

for i, v in enumerate(logic._histogram_data['voltage']):
    new_freq = voltage_to_freq.get(round(v, 4), np.nan)
    if not np.isnan(new_freq):
        logic._histogram_data['frequency'][i] = new_freq
        if v in logic._scan_results:
            logic._scan_results[v]['frequency'] = new_freq

import copy
logic.sigUpdatePlot.emit(copy.deepcopy(logic._histogram_data))
print("GUI plot updated with corrected frequencies.")